# Ablation 4 — MCQ Template Comparison

**Thesis Table**: Section 5.x — Prompt template sensitivity

Tests three MCQ prompt templates on the same Reactor 1 signals:

| Template | Description |
|----------|-------------|
| MCQ_CATEGORIES | Standard 7-class single-answer template |
| MCQ_STALE_FOCUSED | Frozen/variance-collapse focused (no phase, no intermittent) |
| MCQ_MULTI | Multi-label — list all applicable categories |

Key questions:
- Does a frozen-focused template improve C/G detection?
- Does multi-label hurt or help — does ChatTS abuse the flexibility?
- Which template performs best per anomaly type?

In [ ]:
import sys
sys.path.insert(0, '../..')
from dotenv import load_dotenv
load_dotenv('../../.env')

import torch
from src.data.timeseer_client import fetch_series_api, list_series_api
from src.data.ground_truth import GROUND_TRUTH
from src.models.chatts_loader import load_model
from src.prescreener.analyze import hybrid_analyze
from src.inference.chatts_infer import ask_chatts_chunk
from src.parsing.extract import extract_category
from src.evaluation.metrics import compute_metrics
from src.prompts.templates import MCQ_CATEGORIES, MCQ_STALE_FOCUSED, MCQ_MULTI

AREA = 'Reactor 1'

TEMPLATES = {
    'MCQ_CATEGORIES':    MCQ_CATEGORIES,
    'MCQ_STALE_FOCUSED': MCQ_STALE_FOCUSED,
    'MCQ_MULTI':         MCQ_MULTI,
}

print('Imports OK')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
model, tokenizer, processor = load_model('ChatTS-14B')
tags = list_series_api(AREA)
print(f'{len(tags)} signals in {AREA}')

# Cache hybrid_analyze outputs (pre-screen is template-independent)
analyze_cache = {}
for tag in tags:
    vals, idx = fetch_series_api(tag, AREA)
    if vals is None:
        continue
    chunk, question, tname, detected, chunk_desc = hybrid_analyze(vals, idx, tag)
    analyze_cache[tag] = (chunk, question, tname, detected, chunk_desc)
print(f'Pre-screened {len(analyze_cache)} signals')

In [ ]:
all_results = {}

for tmpl_name, tmpl_text in TEMPLATES.items():
    print(f'\n── Template: {tmpl_name} ──')
    results = []

    for tag, (chunk, question, tname, detected, chunk_desc) in analyze_cache.items():
        # Replace MCQ block in question with this template
        # The MCQ block starts at "Select the single best label:"
        base_q = question.split('Select the single best label:')[0].split(
                    'Select all applicable labels:')[0].split(
                    'Stale data or variance collapse?')[0].rstrip()
        full_q = base_q + '\n\n' + tmpl_text

        torch.cuda.empty_cache()
        answer = ask_chatts_chunk(
            chunk, full_q,
            model=model, tokenizer=tokenizer, processor=processor
        )
        cat, lbl = extract_category(answer, detected)
        gt = GROUND_TRUTH.get(tag, '?')
        results.append({'Tag': tag, 'gt': gt, 'Category': cat, 'Label': lbl,
                        'Template': tmpl_name})
        print(f'  {tag:<30} → {cat}) {lbl}  (GT: {gt})')

    all_results[tmpl_name] = results

In [ ]:
print('\n' + '='*70)
print(f'MCQ TEMPLATE COMPARISON — {AREA} — ChatTS-14B')
print('='*70)

for tmpl_name, results in all_results.items():
    evaluated = [r for r in results if r['gt'] != '?']
    preds = [r['Category'] for r in evaluated]
    gts   = [r['gt']       for r in evaluated]
    m = compute_metrics(preds, gts)
    correct = sum(p == g for p, g in zip(preds, gts))
    print(f'{tmpl_name:<22}  Acc={m["accuracy"]:.3f} ({correct}/{len(gts)})  '
          f'MacroF1={m["f1"]:.3f}')

print('\n' + '-'*70)
print(f'{"Tag":<30} {"GT":<4}', end='')
for t in TEMPLATES:
    print(f'  {t[:16]:<16}', end='')
print()
print('-'*70)

for tag in analyze_cache:
    gt = GROUND_TRUTH.get(tag, '?')
    if gt == '?':
        continue
    print(f'{tag:<30} {gt:<4}', end='')
    for tmpl_name in TEMPLATES:
        res = next((r for r in all_results[tmpl_name] if r['Tag'] == tag), None)
        pred = res['Category'] if res else '?'
        mark = '✓' if pred == gt else '✗'
        print(f'  {pred}{mark:<15}', end='')
    print()

In [ ]:
# Per-class breakdown
label_names = {'A':'Drift','B':'Spikes','C':'Frozen','D':'Phase','E':'Clean',
               'G':'VarCollapse','L':'Intermittent'}

all_gts_list = [GROUND_TRUTH.get(t, '?') for t in analyze_cache]
unique_classes = sorted(set(g for g in all_gts_list if g != '?'))

print('\nPer-class accuracy per template:')
print(f'{"Class":<18}', end='')
for t in TEMPLATES:
    print(f'  {t[:16]:<16}', end='')
print()
print('-'*70)

for cls in unique_classes:
    name = label_names.get(cls, cls)
    print(f'{cls}) {name:<15}', end='')
    for tmpl_name in TEMPLATES:
        cls_results = [r for r in all_results[tmpl_name] if r['gt'] == cls]
        if not cls_results:
            print(f'  {"N/A":<16}', end='')
            continue
        correct = sum(r['Category'] == cls for r in cls_results)
        print(f'  {correct}/{len(cls_results)} ({100*correct/len(cls_results):.0f}%)       ', end='')
    print()